# Data Quality Monitoring System for E-Commerce Operations

## Project Overview

This project presents the development of a Data Quality Monitoring System using the **Olist Brazilian E-Commerce Public Dataset**. The objective is to simulate a real-world business analytics workflow by identifying, assessing, and improving the quality of transactional data before it is used for reporting and decision-making.

The project follows an end-to-end data analytics pipeline, beginning with data preparation in **Python**, followed by data modelling and quality analysis in **PostgreSQL**, and concluding with interactive **Power BI** dashboards that monitor key data quality metrics and business performance indicators.

Throughout the project, data quality dimensions such as **completeness, accuracy, consistency, validity, uniqueness, and timeliness** are evaluated. The datasets are cleaned, standardized, validated, and transformed into analysis-ready data to support reliable business intelligence and operational reporting.

This project demonstrates practical skills in data cleaning, exploratory data analysis, SQL development, data quality assessment, and dashboard development while following industry-standard data analytics practices.

This script focuses on the preparation of the Olist order items dataset as part of the Data Quality Monitoring System for E-Commerce Operations. The cleaned dataset supports transactional analysis by linking orders, products, and sellers, forming the foundation for sales, revenue, and product performance analysis in SQL and Power BI.


### Import the Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re 
import psycopg2
from sqlalchemy import create_engine 
from pathlib import Path

### Load The Dataset

In [3]:
order_items_df = pd.read_csv(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce\01_datasets\raw\olist_order_items_dataset.csv")

## **ORDER ITEMS DATASET**

### 1. Data Inspection

In [ ]:
# First 5 rows of the order items dataset
order_items_df.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [ ]:
# Check the number of rows and columns in the order items dataset
order_items_df.shape

(112650, 7)

In [ ]:
# Check the data type of each column in the dataset
order_items_df.dtypes

order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object

The order_items dataset contains seven attributes describing the products purchased within each order. Six columns are stored using appropriate data types. The shipping_limit_date column is currently stored as an object despite representing date and time values. During the data preparation phase, this column will be converted to a datetime data type to support temporal analysis, SQL date functions, and Power BI time intelligence features. No additional data type changes are required.

In [12]:
order_items_df.describe()

,order_item_id,price,freight_value
count,112650.000000,112650.000000,112650.000000
mean,1.197834,120.653739,19.990320
std,0.705124,183.633928,15.806405
min,1.000000,0.850000,0.000000
25%,1.000000,39.900000,13.080000
50%,1.000000,74.990000,16.260000
75%,1.000000,134.900000,21.150000
max,21.000000,6735.000000,409.680000


The descriptive statistics indicate that the numerical fields are generally consistent with the expected characteristics of an e-commerce transactional dataset. The order_item_id values range from **1** to **21**, with most records having a value of 1, suggesting that the majority of customer orders consist of a single item. This distribution aligns with typical online purchasing behaviour and does not indicate a data quality concern.

The price column contains values ranging from **0.85 to 6,735.00**. While the maximum price is substantially higher than both the median **(74.99)** and mean **(120.65)**, this difference alone does not indicate invalid data. E-commerce platforms often sell products across multiple price segments, from inexpensive household items to premium electronics and furniture. Therefore, these high-value transactions should be validated against the corresponding product records before being classified as anomalies.

Similarly, the freight_value column ranges from **0.00 to 409.68**. The presence of zero freight charges may represent legitimate business scenarios such as free shipping promotions or seller-funded delivery rather than missing or incorrect data. Higher freight costs may also be expected for large, heavy, or long-distance shipments. These values should therefore be validated against the related order and product information instead of being treated as outliers.

Overall, no evidence of invalid numerical values is apparent from the descriptive statistics alone. However, the unusually high product prices and freight charges will be examined further during the validation phase to confirm that they represent legitimate business transactions rather than data entry errors.

In [ ]:
# The number of missing values in each column of the order items dataset
order_items_df.isna().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

The missing value assessment indicates that the **order_items** dataset is **100% complete** across all seven attributes. Every order item record contains the required order identifier, product identifier, seller identifier, shipping limit date, item price, and freight cost. No null values were identified that would compromise the integrity of transactional analysis or the relationships between the orders, products, and sellers tables.

In [17]:
# The number of duplicate values in each column of the order items dataset
for column in order_items_df.columns:
    print(f"{column}: {order_items_df[column].duplicated().sum()}")

order_id: 13984
order_item_id: 112629
product_id: 79699
seller_id: 109555
shipping_limit_date: 19332
price: 106682
freight_value: 105651


In [18]:
# Verifying that every item within an order has a unique sequence number and that there are no duplicate order lines.
order_items_df.duplicated(
    subset=["order_id", "order_item_id"]
).sum()

np.int64(0)

The duplicate value assessment identified repeated values across several columns. These results were evaluated against the business purpose of each attribute to determine whether they represent expected transactional relationships or potential data quality issues.

The order_id column contains **13,984** duplicate values, which is expected because a single customer order can contain multiple products. Each product purchased within the same order is recorded as a separate order item, resulting in repeated order identifiers.

The order_item_id column contains **112,629** duplicate values. This is also expected because the item sequence restarts for every new order. For example, most orders begin with order_item_id = 1, making duplicate values a normal characteristic of the dataset. The column is not globally unique and should only be interpreted together with order_id.

The product_id and seller_id columns contain **79,699 and 109,555** duplicate values, respectively. These duplicates reflect normal business activity, as the same product may be purchased multiple times and the same seller may fulfil many different customer orders.

Similarly, duplicate values in **shipping_limit_date, price, and freight_value** are expected because multiple transactions can legitimately share the same shipping deadline, selling price, or freight charge.

Neither order_id nor order_item_id is unique on its own, but the combination should be. It is confirmed that every item within an order has a unique sequence number and that there are no duplicate order lines.

The observed duplicates reflect the transactional nature of the Olist order items dataset rather than data quality defects. Consequently, **no duplicate values should be removed** based solely on this analysis.

### 2. Data Profiling

In [20]:
# # Check the number of unique values in each column of the order items dataset.
unique_summary = pd.DataFrame({
    "Column": order_items_df.columns,
    "Unique Values": order_items_df.nunique().values
})

unique_summary

,Column,Unique Values
0,order_id,98666
1,order_item_id,21
2,product_id,32951
3,seller_id,3095
4,shipping_limit_date,93318
5,price,5968
6,freight_value,6999


The unique value assessment indicates that the order_items dataset is structurally consistent with the expected design of a transactional e-commerce table. The number of unique values observed in each column aligns with the intended purpose of the attributes, with repeated order_id, product_id, seller_id, price, and freight_value values reflecting legitimate transactional relationships rather than data quality issues. Similarly, the order_item_id column contains only 21 unique values because the item sequence restarts for each order. Based on this assessment, no abnormalities requiring data cleaning were identified, and the uniqueness of the dataset is consistent with the expected Olist data model.

In [22]:
# Check the frequency distribution of categorical columns in the order items dataset.

categorical_columns = [
    "order_id",
    "product_id",
    "seller_id"
]

for column in categorical_columns:
    print(f"\n{'=' * 60}")
    print(f"Value Counts: {column}")
    print("=" * 60)
    print(order_items_df[column].value_counts().head(20))


Value Counts: order_id
order_id
8272b63d03f5f79c56e9e4120aec44ef    21
1b15974a0141d54e36626dca3fdc731a    20
ab14fdcfbe524636d65ee38360e22ce8    20
9ef13efd6949e4573a18964dd1bbe7f5    15
428a2f660dc84138d969ccd69a0ab6d5    15
9bdc4d4c71aa1de4606060929dee888c    14
73c8ab38f07dc94389065f7eba4f297a    14
37ee401157a3a0b28c9c6d0ed8c3b24b    13
2c2a19b5703863c908512d135aa6accc    12
c05d6a79e55da72ca780ce90364abed9    12
3a213fcdfe7d98be74ea0dc05a8b31ae    12
637617b3ffe9e2f7a2411243829226d0    12
af822dacd6f5cff7376413c03a388bb7    12
7f2c22c54cbae55091a09a9653fd2b8a    11
71dab1155600756af6de79de92e712e3    11
6c355e2913545fa6f72c40cbca57729e    11
5a3b1c29a49756e75f1ef513383c0c12    11
a483ffe0ce133740ab12ebcba8a3ccf9    10
9aec4e1ae90b23c7bf2d2b3bfafbd943    10
ca3625898fbd48669d50701aba51cd5f    10
Name: count, dtype: int64

Value Counts: product_id
product_id
aca2eb7d00ea1a7b8ebd4e68314663af    527
99a4788cb24856965c36a24e339b6058    488
422879e10f46682990de24d770e7f83d    484
389d

The value count assessment indicates that the frequency distribution of the order_id column is consistent with the expected structure of the Olist order items dataset. Multiple occurrences of the same order_id are expected because a single customer order can contain multiple products, with each product recorded as a separate order item.

In [24]:
# Check the date range for all date columns in the order items dataset.
date_columns = [
    "shipping_limit_date"
]

for column in date_columns:
    print(f"\n{column}")
    print(f"Earliest: {order_items_df[column].min()}")
    print(f"Latest: {order_items_df[column].max()}")


shipping_limit_date
Earliest: 2016-09-19 00:15:34
Latest: 2020-04-09 22:35:08


The shipping_limit_date column spans from 19 September 2016 to 9 April 2020, indicating a complete timeline without any immediately apparent invalid or unrealistic timestamp values. The date range is internally consistent for the Olist dataset and does not suggest formatting errors or abnormal date values.

In [25]:
# Check for leading and trailing whitespace in all text columns.
text_columns = [
    "order_id",
    "product_id",
    "seller_id"
]

for column in text_columns:
    whitespace = (
        order_items_df[column] != order_items_df[column].str.strip()
    ).sum()

    print(f"{column}: {whitespace}")

order_id: 0
product_id: 0
seller_id: 0


In [27]:
# Check for empty strings in all text columns.
for column in text_columns:
    empty = (order_items_df[column] == "").sum()
    print(f"{column}: {empty}")

order_id: 0
product_id: 0
seller_id: 0


In [ ]:
# Check for lowercase values in all text columns.
for column in text_columns:
    lowercase = (
        order_items_df[column]
        == order_items_df[column].str.lower()
    ).all()

    print(f"{column}: {lowercase}")

order_id: True
product_id: True
seller_id: True


In [ ]:
# Check for negative and zero values in all numeric columns.
numeric_columns = [
    "order_item_id",
    "price",
    "freight_value"
]

for column in numeric_columns:
    print(f"\n{column}")
    print(f"Negative values: {(order_items_df[column] < 0).sum()}")
    print(f"Zero values: {(order_items_df[column] == 0).sum()}")


order_item_id
Negative values: 0
Zero values: 0

price
Negative values: 0
Zero values: 0

freight_value
Negative values: 0
Zero values: 383


The text quality assessment identified no leading or trailing whitespace, no empty string values, and consistent lowercase formatting across all text columns, indicating that the textual data is standardized and does not require cleaning. Validation of the numerical columns also found no negative values, confirming that the recorded prices and freight charges are logically valid. While **383 records contain a freight value of zero**, this does not necessarily indicate a data quality issue, as these records may represent legitimate free shipping promotions or seller-funded delivery. These records should be verified against the related order and payment information before determining whether any corrective action is required. Overall, no immediate data quality issues requiring cleaning were identified during this assessment.

### 3. Data Cleaning

In [ ]:
# Convert the shipping limit date to a datetime data type.
order_items_df["shipping_limit_date"] = pd.to_datetime(
    order_items_df["shipping_limit_date"]
)

In [33]:
# Validate the data type conversion.
print(order_items_df.dtypes)

order_id                       object
order_item_id                   int64
product_id                     object
seller_id                      object
shipping_limit_date    datetime64[ns]
price                         float64
freight_value                 float64
dtype: object


In [ ]:
# Convert text columns to the string data type.
text_columns = [
    "order_id",
    "product_id",
    "seller_id"
]

order_items_df[text_columns] = order_items_df[text_columns].astype("string")

In [35]:
print(order_items_df.dtypes)

order_id               string[python]
order_item_id                   int64
product_id             string[python]
seller_id              string[python]
shipping_limit_date    datetime64[ns]
price                         float64
freight_value                 float64
dtype: object


In [36]:
# Validate the uniqueness of the composite key.
order_items_df.duplicated(
    subset=["order_id", "order_item_id"]
).sum()

np.int64(0)

In [37]:
# Validate the cleaned dataset.

print(order_items_df.info())
print("\nMissing values:")
print(order_items_df.isnull().sum())

print("\nDuplicate rows:")
print(order_items_df.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             112650 non-null  string        
 1   order_item_id        112650 non-null  int64         
 2   product_id           112650 non-null  string        
 3   seller_id            112650 non-null  string        
 4   shipping_limit_date  112650 non-null  datetime64[ns]
 5   price                112650 non-null  float64       
 6   freight_value        112650 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(1), string(3)
memory usage: 6.0 MB
None

Missing values:
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

Duplicate rows:
0


The cleaning and validation process confirmed that the order items dataset required only minor transformations. The shipping_limit_date column was successfully converted to the datetime data type, and all identifier columns were standardized using the pandas string data type. Validation confirmed that the composite key (order_id, order_item_id) remained unique, no missing values were introduced during the cleaning process, and the dataset retained its structural integrity. The dataset is therefore considered clean, validated, and ready for export to the cleaned data repository.

### 3. Export the cleaned dataset

In [38]:
PROJECT_ROOT = Path(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce")

RAW_DATA_PATH = PROJECT_ROOT / "01_datasets" / "raw"
CLEANED_DATA_PATH = PROJECT_ROOT / "01_datasets" / "cleaned"

In [39]:
order_items_df.to_csv(
    CLEANED_DATA_PATH / "olist_order_items_cleaned.csv",
    index=False
)